# MIDI Dataset Statistics

统计不同数据集的 MIDI pitch / velocity 分布，输出直方图和缺失值。


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "MAESTRO v3" \
  --root "/media/mengh/SharedData/zhanh/Dataset/maestro-v3.0.0" \
  --output-dir "figures/midi_dataset_stats" \
  --json-out-dir "stats/midi_sampler"

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "SMD" \
  --root "/media/mengh/SharedData/zhanh/Dataset/SMD" \
  --output-dir "figures/midi_dataset_stats" \
  --json-out-dir "stats/midi_sampler"

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "MAPS (ENSTDkCl + ENSTDkAm)" \
  --root "/media/mengh/SharedData/zhanh/Dataset/MAPS" \
  --patterns "ENSTDkCl/**/*.mid" "ENSTDkAm/**/*.mid" \
  --output-dir "figures/midi_dataset_stats" \
  --json-out-dir "stats/midi_sampler"

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "FrancoisLeducGuitarDataset" \
  --root "/media/mengh/SharedData/zhanh/Dataset/FrancoisLeducGuitarDataset"

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "GOAT" \
  --root "/media/mengh/SharedData/zhanh/Dataset/GOAT" \
  --patterns "**/*.mid" "**/*.midi"

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "GAPS" \
  --root "/media/mengh/SharedData/zhanh/Dataset/GAPS" \
  --patterns "**/*.mid" "**/*.midi"


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "MID-FiLD" \
  --root "/media/mengh/SharedData/zhanh/Dataset/MID-FiLD_dataset"
#   --patterns "train/**/*.mid" "train/**/*.midi"

%run -m data_analysis.analysis.dataset_midi_stats \
  --dataset "MusicNet" \
  --root "/media/mengh/SharedData/zhanh/Dataset/MusicNet/musicnet_midis"

In [ ]:
import mirdata

ROOT = "/media/mengh/SharedData/zhanh/Dataset/guitarset"

ds = mirdata.initialize("guitarset", data_home=ROOT)

try:
    _ = ds.track_ids
except FileNotFoundError:
    ds.download(partial_download=["index"])  # 只补 mirdata 的索引文件

print("n_tracks =", len(ds.track_ids))
print(ds.track_ids[:5])


In [ ]:
track = ds.track("00_BN1-129-Eb_comp")

print(track.player_id, track.style, track.tempo, track.mode)
print(track.jams_path)
print(track.notes.keys())   # 每根弦: E A D G B e

notes_all = track.notes_all
print(notes_all.intervals[:5])   # [start, end]
print(notes_all.pitches[:5])     # MIDI pitch


In [ ]:
import numpy as np

all_pitches = []
all_durations = []

for track_id in ds.track_ids:
    notes = ds.track(track_id).notes_all
    if notes is None:
        continue
    all_pitches.append(notes.pitches)
    all_durations.append(notes.intervals[:, 1] - notes.intervals[:, 0])

all_pitches = np.concatenate(all_pitches)
all_durations = np.concatenate(all_durations)

print("num_notes =", len(all_pitches))
print("pitch_min,max =", all_pitches.min(), all_pitches.max())
print("duration_mean =", all_durations.mean())


In [ ]:
hop = 0.01
time_scale = np.arange(0, notes_all.intervals[:, 1].max() + hop, hop)
midi_scale = np.arange(1, 128)   # 避开 0

roll = notes_all.to_matrix(time_scale, "s", midi_scale, "midi")
print(roll.shape)   # [time, pitch]


In [ ]:
track = ds.track(ds.track_ids[0])

print(type(track))
print(type(track.notes_all))
print(type(track.notes["E"]))
print(type(track.pitch_contours["E"]))
print(type(track.beats))
print(type(track.multif0))


In [ ]:
for name in [x for x in dir(track) if not x.startswith("_")]:
    try:
        print(name, "->", type(getattr(track, name)).__name__)
    except Exception as e:
        print(name, "->", e)


In [ ]:
notes_all.intervals[:10]
# array([
#   [0.0488, 0.4726],
#   [0.0498, 0.5026],
#   [0.0527, 0.5113],
#   [0.5200, 0.9380],
#   ...
# ])

In [ ]:
notes_all.pitches[:10]
# array([51.0368, 65.0496, 62.0914, 51.0325, 58.1507, ...])
